In [ ]:
# Localiza la raíz del repositorio subiendo desde donde se ejecute el notebook,
# para no depender de una ruta fija de una máquina concreta.
from pathlib import Path

PROJECT_DIR = Path.cwd().resolve()
while not ((PROJECT_DIR / "data").exists() and (PROJECT_DIR / "notebooks").exists()):
    PROJECT_DIR = PROJECT_DIR.parent

# 1. Importación

Carga de las librerías necesarias y del dataset de features resultante de `02_feature_engineering.ipynb` (`data/processed/mallorca/listings_full_features.csv`).

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv(f"{PROJECT_DIR}/data/processed/mallorca/listings_full_features.csv")
df.shape

(11044, 126)

## 2. Preparación de X e y

Separar identificadores, objetivo (`price`) y features. Convertir las columnas booleanas a 0/1 y decidir qué hacer con los nulos que quedan, ya que una regresión lineal no admite `NaN` directamente.

### 2.1 Identificadores, objetivo y features

Además de los identificadores, hay que excluir de `feature_cols` las columnas calculadas directamente a partir de `price` (`price_log`, `price_per_accommodate`, `price_per_min_night`): si se dejan dentro, el modelo no aprende ningún patrón real, deshace la fórmula y "adivina" el precio exacto. Es fuga de información, igual que en las otras cinco ciudades.

**Fuga corregida más abajo**: `neighbourhood_price_encoded` (creada en `02_feature_engineering.ipynb`) se calculó allí usando todo el dataset, no solo lo que aquí es train. Se arregla en la sección 3.1bis, justo después del split. Por eso `neighbourhood_cleansed` (la columna cruda) sigue en `df` en este punto.

In [3]:
id_cols = ["id", "host_id", "host_profile_id"]
target_col = "price"
leakage_cols = ["price_log", "price_per_accommodate", "price_per_min_night"]
# neighbourhood_cleansed todavía no es una feature (pendiente de la corrección de la
# sección 3.1bis): se excluye de X igual que los identificadores, pero se mantiene en
# df para poder usarla justo después del split.
pending_cols = ["neighbourhood_cleansed"]
feature_cols = [c for c in df.columns if c not in id_cols + [target_col] + leakage_cols + pending_cols]

X = df[feature_cols].copy()
y = df[target_col].copy()

X.shape, y.shape

((11044, 118), (11044,))

### 2.2 Booleanas a 0/1

87 de las 118 features son booleanas (los one-hot y los flags) — proporción más alta que en Málaga (43 de 74), por los 53 dummies de `district_*` frente a los 11 de Málaga. `scikit-learn` las admite tal cual, pero se convierten a `int` de forma explícita.

In [4]:
bool_cols = X.select_dtypes(include="bool").columns
X[bool_cols] = X[bool_cols].astype(int)
len(bool_cols)

87

### 2.3 Nulos restantes

`review_scores_rating`, `listing_age_days` y `days_since_last_review` son `NaN` en las 1.850 filas sin reviews todavía (`has_reviews == False`). Para este baseline se imputan con la mediana; un modelo de árboles en `04_model_training.ipynb` podrá trabajar con el `NaN` directamente.

In [5]:
null_cols = X.columns[X.isnull().any()].tolist()
print(null_cols)

X[null_cols] = X[null_cols].fillna(X[null_cols].median())
X.isnull().sum().sum()

['review_scores_rating', 'listing_age_days', 'days_since_last_review']


np.int64(0)

`X` queda con 118 columnas numéricas sin nulos, e `y` es `price` sin transformar (el logaritmo se aplica más adelante, solo para el modelo de la sección 6).

## 3. Train/test split

Reservar un conjunto de test antes de tocar nada más, y guardarlo en `data/processed/mallorca/` para que `04_model_training.ipynb` y `05_model_evaluation.ipynb` partan del mismo split.

### 3.1 Dividir

80/20, con `random_state` fijo. Se estratifica por `room_type`: en la EDA se vio que `Hotel room` (0,24%, 27 anuncios) tiene muestra muy escasa en Mallorca. Un split aleatorio sin más podría dejar a esa categoría con muy pocas filas en test por puro azar. A diferencia de Málaga (que también tenía que vigilar `Shared room`), aquí `room_type` solo tiene tres valores: no existe ningún anuncio de `Shared room` en la isla.

In [6]:
room_type_cols = [c for c in df.columns if c.startswith("room_type_")]
room_type_for_stratify = df[room_type_cols].idxmax(axis=1)

X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    X, y, df, test_size=0.2, random_state=42, stratify=room_type_for_stratify
)

X_train.shape, X_test.shape

((8835, 118), (2209, 118))

### 3.1bis Corregir la fuga de `neighbourhood_price_encoded`

Mismo cálculo que en `02_feature_engineering.ipynb` (media de `price` por `neighbourhood_cleansed`, suavizada con la media global y `smoothing=10`), pero ahora solo con `df_train`. El mapa aprendido en train se aplica tal cual a `df_test` (un municipio de test que no apareciera en train recibiría la media global de train como respaldo).

Con esto corregido, `neighbourhood_cleansed` ya cumplió su función y se descarta de `df_train`/`df_test`. Como esta misma columna también generó los 53 dummies `district_*` en `02_feature_engineering.ipynb` (sección 5.2, al no existir un nivel de distrito separado en Mallorca), esos dummies no se ven afectados por esta corrección: se calcularon directamente por fila, sin agregación sobre todo el dataset, así que no tenían fuga que corregir.

In [7]:
smoothing = 10
global_mean_price_train = y_train.mean()
neigh_stats_train = df_train.groupby("neighbourhood_cleansed")["price"].agg(["mean", "count"])
smoothed_mean_train = (
    neigh_stats_train["count"] * neigh_stats_train["mean"] + smoothing * global_mean_price_train
) / (neigh_stats_train["count"] + smoothing)

df_train["neighbourhood_price_encoded"] = df_train["neighbourhood_cleansed"].map(smoothed_mean_train)
df_test["neighbourhood_price_encoded"] = (
    df_test["neighbourhood_cleansed"].map(smoothed_mean_train).fillna(global_mean_price_train)
)

print("municipios de test no vistos en train:", df_test["neighbourhood_cleansed"].map(smoothed_mean_train).isnull().sum())

# X_train/X_test ya tenían la versión con fuga (calculada en la sección 2.1 antes del
# split): se sincronizan con el valor corregido de df_train/df_test.
X_train["neighbourhood_price_encoded"] = df_train["neighbourhood_price_encoded"]
X_test["neighbourhood_price_encoded"] = df_test["neighbourhood_price_encoded"]

df_train = df_train.drop(columns=["neighbourhood_cleansed"])
df_test = df_test.drop(columns=["neighbourhood_cleansed"])

df_train[["neighbourhood_price_encoded"]].describe()

municipios de test no vistos en train: 0


,neighbourhood_price_encoded
count,8835.000000
mean,618.660012
std,102.648318
min,417.459768
25%,570.008641
50%,595.382065
75%,672.804633
max,1008.948029


Los 53 municipios de Mallorca aparecen todos en train (incluso `Santa Eugènia` y `Estellencs`, los más pequeños, empatados con solo 10 anuncios cada uno en todo el dataset), ningún municipio de test se queda sin mapa — a diferencia de lo que cabría temer con tantas categorías y algunas tan minoritarias.

### 3.2 Guardar el split

Se guarda `df_train`/`df_test` ya con `neighbourhood_price_encoded` corregido, pero antes de la imputación y la conversión de booleanas de la sección 2, para que `04_model_training.ipynb` y `05_model_evaluation.ipynb` puedan decidir su propio tratamiento.

In [8]:
df_train.to_csv(f"{PROJECT_DIR}/data/processed/mallorca/listings_train.csv", index=False)
df_test.to_csv(f"{PROJECT_DIR}/data/processed/mallorca/listings_test.csv", index=False)

## 4. Baseline ingenuo

Un modelo trivial (predecir siempre la media, la mediana, o la mediana por una variable de tamaño) como suelo mínimo: cualquier modelo real tiene que superar esto para que merezca la pena.

### 4.1 Predecir siempre la media

Por definición, un modelo que siempre predice la media del train tiene R² ≈ 0 sobre el test. Sirve como punto cero.

In [9]:
global_mean = y_train.mean()
pred_mean = np.full(len(y_test), global_mean)

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_mean)))
print("MAE:", mean_absolute_error(y_test, pred_mean))
print("R2:", r2_score(y_test, pred_mean))

RMSE: 896.6185143320697
MAE: 456.0529078388227
R2: -0.0007704104616692575


### 4.2 Predecir siempre la mediana

Dado el sesgo de `price` (skew 6,71 en la EDA), la mediana debería ser un mejor "valor típico" que la media.

In [10]:
global_median = y_train.median()
pred_median = np.full(len(y_test), global_median)

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_median)))
print("MAE:", mean_absolute_error(y_test, pred_median))
print("R2:", r2_score(y_test, pred_median))

RMSE: 929.841340075353
MAE: 402.9544182888185
R2: -0.07630845718118029


El MAE mejora (402,95€ frente a 456,05€ con la media), pero el R² empeora (-0,08 frente a ~0,00): el R² compara contra la media por definición, así que cualquier predicción distinta puede bajarlo aunque sea mejor en otros términos. Ambos MAE son, con diferencia, los más altos de las seis ciudades — refleja directamente la dispersión de precio real de Mallorca (media ~625€, con villas de hasta 22.260€ todavía dentro del rango tras la limpieza de la EDA), varias veces mayor que la de cualquier ciudad del proyecto.

### 4.3 Predecir la mediana según `accommodates`

`accommodates` es una de las variables con correlación lineal más fuerte con `price` en Mallorca (Pearson 0,43, visto en feature engineering, aunque por detrás de `bathrooms` con 0,45). Un baseline algo menos ingenuo: mediana por cada valor de `accommodates`, calculada solo con train.

In [11]:
train_medians_by_accommodates = X_train.assign(price=y_train).groupby("accommodates")["price"].median()

pred_accommodates = X_test["accommodates"].map(train_medians_by_accommodates).fillna(global_median)

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_accommodates)))
print("MAE:", mean_absolute_error(y_test, pred_accommodates))
print("R2:", r2_score(y_test, pred_accommodates))

RMSE: 849.6139123878344
MAE: 332.2484449977365
R2: 0.10140854982515035


Mejora, pero mucho más discreta que en las demás ciudades: RMSE de 929,8 a 849,6, MAE de 402,95 a 332,25, R² de -0,08 a solo **0,10**. En Mallorca, `accommodates` por sí solo explica bastante menos varianza que en el resto del proyecto: el precio depende más del *tipo* de propiedad (villa vs. piso, visto en la EDA/feature engineering) que del tamaño en sí, así que agrupar solo por tamaño deja fuera la fuente de variación más importante.

## 5. Métricas de evaluación

Definir aquí las métricas que se van a usar de forma consistente en todo el modelado (RMSE, MAE, R², MAPE).

### 5.1 Función `evaluate`

Se añade una cuarta métrica, MAPE, el error medio en porcentaje sobre el precio real. Todo se empaqueta en una función para no repetir el código en cada modelo.

In [12]:
def evaluate(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {"modelo": name, "RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape}

### 5.2 Tabla comparativa de los baselines

In [13]:
rows = [
    evaluate(y_test, pred_mean, "Media"),
    evaluate(y_test, pred_median, "Mediana"),
    evaluate(y_test, pred_accommodates, "Mediana por accommodates"),
]

results = pd.DataFrame(rows).set_index("modelo")
results.round(2)

,RMSE,MAE,R2,MAPE
modelo,,,,
Media,896.62,456.05,-0.00,123.30
Mediana,929.84,402.95,-0.08,71.57
Mediana por accommodates,849.61,332.25,0.10,48.02


El MAPE sale muy alto (48-123%), el más alto de las seis ciudades: hay anuncios muy baratos (`price` mínimo 28,41€, visto en la EDA) donde un error de pocos cientos de euros ya es un porcentaje enorme, y la enorme dispersión de precios (de 28€ a 22.260€) hace que cualquier predicción única falle mucho para una punta o la otra. Conviene fiarse más de RMSE/MAE/R² que del MAPE por sí solo, todavía más que en las otras ciudades.

## 6. Baseline real: regresión lineal

Un primer modelo simple e interpretable, entrenado sobre `price_log` por el sesgo ya visto en la EDA.

### 6.1 Entrenar

Se entrena sobre `log1p(price)`. Las predicciones se deshacen con `expm1` antes de evaluar.

In [14]:
y_train_log = np.log1p(y_train)

model = LinearRegression()
model.fit(X_train, y_train_log)

pred_log = model.predict(X_test)
pred_lr = np.expm1(pred_log)

/Users/yagocoll/Documents/Master/Airbnb/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/yagocoll/Documents/Master/Airbnb/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/yagocoll/Documents/Master/Airbnb/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


### 6.2 Evaluar

In [15]:
results.loc["Regresión lineal (log)"] = evaluate(y_test, pred_lr, "Regresión lineal (log)")
results.round(2)

,RMSE,MAE,R2,MAPE
modelo,,,,
Media,896.62,456.05,-0.00,123.30
Mediana,929.84,402.95,-0.08,71.57
Mediana por accommodates,849.61,332.25,0.10,48.02
Regresión lineal (log),699.54,267.99,0.39,39.64


**R²=0,39, MAE≈268€**: un ajuste relativo comparable al de Málaga (0,42) o Sevilla, pese al mercado mucho más disperso y complejo (villas vs. pisos, 53 municipios en vez de 11). El MAE en euros (268€) es, como era de esperar, el más alto de las seis ciudades con diferencia: la escala absoluta de precios en Mallorca (media ~625€) no tiene comparación con ninguna ciudad del proyecto. Aun así, el salto sobre el mejor baseline ingenuo es enorme: R² de 0,10 a 0,39, MAE de 332€ a 268€ — la regresión lineal sí aprovecha el resto de variables (tipo de propiedad, municipio, amenities...) que el baseline por `accommodates` dejaba fuera.

### 6.3 Un aviso a tener en cuenta

Al entrenar (sección 6.1) aparecen avisos de `numpy` (`divide by zero`, `overflow`... `encountered in matmul`), igual que en las otras ciudades.

In [16]:
import numpy.linalg as la

la.cond(X_train.values)

np.float64(2.4399013726667596e+19)

Un número de condición altísimo (2,4·10¹⁹, el más alto de las seis ciudades) indica una matriz muy mal condicionada: `X` incluye a propósito tanto la versión bruta como la versión `_log` de varias variables (`bedrooms`/`bedrooms_log`...) y varias codificaciones categóricas que se solapan. Aquí se suma, mucho más que en Málaga, la redundancia entre `district_*` (53 columnas) y `neighbourhood_price_encoded`, calculados sobre la misma columna — con 53 categorías en vez de 11, el solape es proporcionalmente mucho mayor. A esto se añade el VIF ya elevado de `accommodates`/`bedrooms` (sección 4 de `02_feature_engineering.ipynb`). Esto no invalida las métricas (`scikit-learn` resuelve con SVD, sin `NaN`/`inf` en las predicciones), pero sí impide interpretar los coeficientes uno a uno. Se deja igual que en las otras ciudades para `04_model_training.ipynb` (un modelo de árboles no depende de esto).

## 7. Conclusiones

Resumen de los resultados del baseline.

### Resultados

| Modelo | RMSE | MAE | R² | MAPE |
|---|---|---|---|---|
| Media | 896.62 | 456.05 | -0.00 | 123.30 |
| Mediana | 929.84 | 402.95 | -0.08 | 71.57 |
| Mediana por `accommodates` | 849.61 | 332.25 | 0.10 | 48.02 |
| Regresión lineal (log) | 699.54 | 267.99 | 0.39 | 39.64 |

Cada paso mejora sobre el anterior: agrupar por `accommodates` ya recorta el MAE de 456€ a 332€, y el modelo real con las 118 variables lo baja a 268€, con un R² de 0,39 — comparable al de Málaga/Sevilla pese a un mercado bastante más heterogéneo (villas, pisos y 53 municipios en vez de un puñado de distritos).

### Tres problemas encontrados y cómo se trataron

- **Fuga de información directa**: `price_log`, `price_per_accommodate` y `price_per_min_night` estaban calculadas a partir de `price` y se habían colado como features. Se excluyeron en la sección 2, igual que en las otras cinco ciudades.
- **Fuga más leve, ahora corregida**: `neighbourhood_price_encoded` se calculaba con todo el dataset. Se corrige en la sección 3.1bis, recalculándola solo con `df_train`. Los 53 dummies `district_*`, calculados sobre la misma columna, no tenían este problema por construirse fila a fila.
- **Muestra escasa en `Hotel room`** (27 anuncios sobre 11.044): el split estratifica por `room_type`.
- **Multicolinealidad severa** en la regresión lineal, la más alta de las seis ciudades (número de condición 2,4·10¹⁹): por la versión bruta y `_log` de varias variables a la vez, el VIF ya elevado de `accommodates`/`bedrooms`, y la redundancia `district_*`/`neighbourhood_price_encoded` amplificada por tener 53 categorías en vez de 11-19. No afecta a las métricas de predicción, pero impide interpretar los coeficientes uno a uno.

### Una diferencia real con las ciudades, no un error de pipeline

El MAE en euros de Mallorca (268€) es varias veces el de cualquier otra ciudad del proyecto, pero el R² (0,39) queda en línea con Málaga/Sevilla. No es una contradicción: R² mide varianza explicada (relativa a la propia dispersión de `price` en Mallorca, mucho mayor que en cualquier ciudad), mientras que MAE mide error típico en euros (absoluto). Un mercado con villas de hasta 22.260€/noche conviviendo con habitaciones de 28€ tiene, por construcción, un error absoluto típico mayor que un mercado de pisos urbanos entre 20€ y 1.000€, aunque el modelo explique una fracción similar de la varianza relativa en ambos casos.

### El listón para `04_model_training.ipynb`

Cualquier modelo más complejo (Random Forest, HistGradientBoosting...) tiene que superar claramente **R²=0,39 / MAE≈268€**. Un modelo de árboles no necesita la imputación por mediana de la sección 2.3, no le afecta la multicolinealidad de la sección 6.3 (la más severa de las seis ciudades), y podría beneficiarse especialmente de `distance_to_center_km`, cuya relación con `price` en Mallorca no es monótona (visto en `02_feature_engineering.ipynb`, sección 8.1): un modelo lineal no puede aprovechar una forma de "U", pero un modelo de árboles sí.

### Lo que queda guardado

`listings_train.csv` y `listings_test.csv` en `data/processed/mallorca/`, con el mismo split (80/20, `random_state=42`, estratificado por `room_type`) y `neighbourhood_price_encoded` ya corregido, para que `04_model_training.ipynb` y `05_model_evaluation.ipynb` trabajen sobre las mismas filas y los resultados sean comparables entre notebooks.